# AeroRL — train on Kaggle

**Training status (September 7): a replacement imitation-trained neural policy now passes 64/64 hover and 64/64 single-gate validation flights. See `notebooks/POLICY_REPAIR.md` for local models, replays and commands. The original PPO campaign below lost hover after stage 1; do not blindly resume it or use its resume path for the replacement actor.**

This notebook trains an experimental PPO pilot using exact simulated state. Start with **Internet on, accelerator None**. CPU training is intentional; physics is currently Python/NumPy. No browser or display server is required.

1. Upload `aerorl-kaggle-source.zip` as a private Kaggle dataset and attach it using Add Input.
2. Import this notebook into Kaggle.
3. Run cells in order. The smoke test must pass before training.
4. Save a notebook version with outputs and download the final run archive before ending the session.

The seven-stage curriculum progresses automatically from hover to one, three, then ten ordered gates. No trained-success claim is made: initial flights are expected to fail. This notebook's diagnostic NPZ traces have their own format; browser replay packaging and release qualification are separate remaining milestones.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
SOURCE = WORK / "aerorl-source"
SOURCE_INPUT = None  # Optional exact zip path or extracted source directory
matches = [Path(SOURCE_INPUT)] if SOURCE_INPUT else sorted(INPUT.rglob("aerorl-kaggle-source.zip"))
if not matches:
    matches = [p.parent.parent for p in INPUT.rglob("requirements-kaggle.lock") if (p.parent.parent / "training" / "envs" / "drone.py").exists()]
if len(matches) > 1 and all(p.is_file() for p in matches):
    import hashlib
    if len({hashlib.sha256(p.read_bytes()).hexdigest() for p in matches}) == 1:
        matches = matches[:1]
if len(matches) != 1:
    raise RuntimeError(f"Set SOURCE_INPUT to one source zip or extracted folder. Found: {matches}")
SOURCE.mkdir(parents=True, exist_ok=True)
if matches[0].is_dir():
    shutil.copytree(matches[0], SOURCE, dirs_exist_ok=True)
else:
    with zipfile.ZipFile(matches[0]) as archive:
        for member in archive.infolist():
            destination = (SOURCE / member.filename).resolve()
            if not destination.is_relative_to(SOURCE.resolve()):
                raise RuntimeError("Unsafe archive path")
        archive.extractall(SOURCE)
SOURCE_ARCHIVE = Path(shutil.make_archive(str(WORK / "aerorl-kaggle-source"), "zip", SOURCE))
print("Source:", SOURCE)


## Isolated, pinned CPU runtime

The setup downloads Python 3.11.14 and installs a hash-locked Linux CPU environment under `/tmp/aerorl-venv`. It leaves Kaggle's kernel packages alone. The official PyTorch CPU wheel is pinned in the lock. Internet is required for this setup; the training itself runs offline. The runtime and package cache stay under `/tmp` and are not included in saved notebook outputs.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.11.32"], check=True)
UV = [sys.executable, "-m", "uv"]
ENV = os.environ.copy()
ENV.update(UV_CACHE_DIR="/tmp/aerorl-uv-cache", UV_PYTHON_INSTALL_DIR="/tmp/aerorl-python",
           PYTHONPATH=str(SOURCE), OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
subprocess.run(UV + ["python", "install", "3.11.14"], env=ENV, check=True)
VENV = Path("/tmp/aerorl-venv")
if not VENV.exists():
    subprocess.run(UV + ["venv", "--python", "3.11.14", str(VENV)], env=ENV, check=True)
PYTHON = VENV / "bin" / "python"
subprocess.run(UV + ["pip", "sync", "--python", str(PYTHON), "--require-hashes",
                    str(SOURCE / "notebooks" / "requirements-kaggle.lock")], env=ENV, check=True)

def run_module(module, *arguments):
    subprocess.run([str(PYTHON), "-u", "-m", module, *map(str, arguments)],
                   cwd=SOURCE, env=ENV, check=True)

run_module("pytest", "training/tests/test_simulator.py", "-q")


## Configuration

`SESSION_STEPS` is additional training for this session. Training stops at a complete rollout, so the requested count can be exceeded by at most 2,047 transitions with one environment. `TOTAL_BUDGET` controls the global learning-rate schedule and must stay unchanged on resume. Leave the defaults for the first real run.

For a later session, attach the previous notebook's saved output, then set `RESUME` to a **checkpoint directory containing model.zip, rng.pt, and campaign.json**, not to model.zip itself. Use the same source dataset, seed, environment count, and total budget. Pick a new OUTPUT directory for each session. Resume restores model, optimizer, RNG streams and curriculum, but restarts unfinished episodes; it is not a bit-for-bit continuation of the interrupted trajectory. Load only your own trusted checkpoints.


In [ ]:
SESSION_STEPS = 1_000_000
TOTAL_BUDGET = 10_000_000
SEED = 101
N_ENVS = 1
RESUME = None  # Example: "/kaggle/input/previous-run/.../checkpoints/step-000001000448"
OUTPUT = WORK / "aerorl-run-001"

assert SESSION_STEPS > 0 and TOTAL_BUDGET > 0
if OUTPUT.exists() and any(OUTPUT.iterdir()) and RESUME is None:
    raise RuntimeError("Choose an unused OUTPUT or explicitly resume a checkpoint.")
OUTPUT.mkdir(parents=True, exist_ok=True)
freeze = subprocess.check_output(UV + ["pip", "freeze", "--python", str(PYTHON)], env=ENV, text=True)
(OUTPUT / "environment.txt").write_text(freeze)
shutil.copy2(SOURCE / "notebooks" / "requirements-kaggle.lock", OUTPUT / "requirements-kaggle.lock")
shutil.copy2(SOURCE_ARCHIVE, OUTPUT / "aerorl-kaggle-source.zip")


## Smoke test and resume check

This uses a separate output and small rollouts; it does not train the real campaign. Passing proves the notebook can collect experience, optimize PPO, record failed attempts, save, and reload. It does not prove the pilot has learned to fly.


In [ ]:
import uuid, time
SMOKE = WORK / ("aerorl-smoke-" + uuid.uuid4().hex[:8])
start = time.monotonic()
run_module("training.learning.train", "--output", SMOKE, "--steps", 256, "--smoke")
first = json.loads((SMOKE / "latest.json").read_text())["checkpoint"]
run_module("training.learning.train", "--output", SMOKE, "--steps", 128, "--smoke", "--resume", first)
last = Path(json.loads((SMOKE / "latest.json").read_text())["checkpoint"])
assert json.loads((last / "campaign.json").read_text())["steps"] == 384
print(f"Training/resume smoke passed in {time.monotonic()-start:.1f}s")


## Start training

Runs for the additional transition count or about nine hours, checked between rollouts. Save a version with outputs for unattended execution. The runner writes atomic checkpoints every 32,768 transitions and at normal exit, keeping the newest two. An abrupt Kaggle shutdown can lose work since the latest complete checkpoint. Evaluation may extend the wall-time boundary; reserve time for it and output saving.

Promotion: every 131,072 transitions, evaluate 64 fixed cases for the current stage. At least 61 successes on two consecutive checks and 131,072 frontier transitions are required to advance. Training resets sample the frontier 80% of the time and a uniformly selected earlier stage 20%. Gates are sampled once at reset, retain labels in route order, and remain fixed throughout that attempt. Stage 6 continues the reliability objective; speed-phase and release evaluation are not enabled in this notebook.


In [ ]:
arguments = ["--output", OUTPUT, "--steps", SESSION_STEPS, "--budget", TOTAL_BUDGET,
             "--seed", SEED, "--n-envs", N_ENVS]
if RESUME:
    arguments += ["--resume", RESUME]
run_module("training.learning.train", *arguments)
LATEST = Path(json.loads((OUTPUT / "latest.json").read_text())["checkpoint"])
print(json.loads((LATEST / "campaign.json").read_text()))


## Learning curves

Episode reward is training reward, not a held-out completion rate. The promotion report records deterministic fixed-case outcomes. TensorBoard logs are also saved in the run directory.


In [ ]:
import matplotlib.pyplot as plt
rows = [json.loads(line) for line in (OUTPUT / "episodes.jsonl").read_text().splitlines()]
if rows:
    window = min(100, len(rows))
    rewards = [r["r"] for r in rows]
    smoothed = [sum(rewards[max(0,i-window+1):i+1])/min(window,i+1) for i in range(len(rows))]
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    axes[0].plot([r["transition"] for r in rows], smoothed)
    axes[0].set_ylabel("Mean reward (last 100)")
    axes[1].plot([r["transition"] for r in rows], [r["stage"] for r in rows], ".", markersize=2)
    axes[1].set(ylabel="Sampled stage", xlabel="Training transitions", yticks=range(7))
    fig.tight_layout()
    fig.savefig(OUTPUT / "learning-curves.png")
    plt.show()
else:
    print("No completed episodes yet.")


## Play an actual training attempt in 3D

Retention keeps the first two completed attempts in each million-transition/stage/outcome bucket, up to approximately 1 GB. These are actual sampled training flights, including failures, rather than recreated deterministic flights. The selector below lists the saved files; set `TRACE` to any listed path. The player includes recorded orientation, time scrubbing, ordered gate outlines, and the flight path. The visual drone arms have the physical 0.2 m arm length; zoom in when needed. Playback may subsample to 600 frames; the NPZ retains all 120 Hz states.


In [ ]:
traces = sorted((OUTPUT / "training-traces").glob("*.npz"))
for path in traces:
    print(path.name)
if not traces:
    raise RuntimeError("No retained training trace yet.")
TRACE = traces[0]
REPLAY_HTML = OUTPUT / "training-attempt.html"
run_module("training.recording.visualize", TRACE, REPLAY_HTML)
from IPython.display import HTML, display
display(HTML(REPLAY_HTML.read_text()))


## Export the actor and preserve outputs

The exported ONNX actor accepts normalized 40-field observations and returns four clipped deterministic commands. A 512-observation parity test must pass. It remains an experimental model, not a browser-ready model bundle or a qualified racing pilot. Keep the source archive with the checkpoint for compatible resumption.


In [ ]:
run_module("training.learning.export", LATEST)
archive = shutil.make_archive(str(WORK / "aerorl-run-001"), "zip", OUTPUT)
print("Download:", archive)
from IPython.display import FileLink
display(FileLink(archive))


## Recorded data and current limits

Each NPZ loads with `numpy.load(path, allow_pickle=False)`. `states` has time plus position, wxyz quaternion, world velocity, body angular velocity, and four actual motor thrusts. `commands` has time, four commanded thrusts, desaturation scale and duration. `observations` contains each 40-field policy input. `actions` has start/end time, clipped actions, collective and body-rate commands. Training files additionally store sampled `raw_actions` and PPO `policy_update` counts. `rewards` contains step totals; `terminal_observation` preserves the final input. JSON metadata contains the full sampled course, seeds, gate-pass events and outcome metrics.

This first training delivery deliberately uses a diagnostic NPZ format and synchronous bounded recording. Full `.aerorl.zip` browser replay manifests, reward-component traces, reservoir retention, multiprocess benchmarking, speed curriculum, final holdout qualification, and TypeScript physics parity remain implementation work. Hover departure is checked at physics endpoints; gate/frame collisions are swept. These limits are documented so these experimental checkpoints are not mistaken for a completed v1 release.

Local validation covers simulator behavior, PPO training/resume, export and diagnostic playback. The notebook must pass its own smoke test on Kaggle; a Kaggle-hosted run has not been performed on your account.
